# Research MVP: a smaller capability check, with Qwen loaded once

Select a **T4 GPU**, then choose **Runtime → Run all** and authorize Drive if prompted. The cells continue automatically through one episode, audit, and ZIP download. This runs one full-history episode: **one room, one ingredient, seed 0**. The Qwen revision, FP16, greedy decoding, `v3` prompt, budget 80, reread price 2, and generation settings are recorded. V3 only prepends appliance rules to the audited `v2-mechanics` prompt; all other run settings match that failed seed-0 attempt. No solution is provided to the model.

The audited small-game run chose the oven for a grilling recipe and lost after four actions. V3 explains stove=fry, oven=roast, toaster=grill without ingredient names or a scripted sequence. Success remains unverified; this is a capability check before memory comparisons. No memory comparison runs automatically.

The previous v2 run spent about **101 seconds loading Qwen and 203 seconds playing**. Keeping weights loaded avoids repeated loading during this live runtime; it does not remove inference cost or survive a Colab disconnect. New timings must be measured.

Open this notebook from the current GitHub link for each code update. REPO_REF=None resolves main once and records the exact commit; set a full SHA to reproduce a frozen run. Each launch preserves a fresh results folder. No additional seeds, retries, or memory comparisons launch automatically.


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys
import uuid

REPO_URL = "https://github.com/sebace05/64610-Memory.git"
REPO_REF = None  # Development: resolve main once. Set a full SHA for a frozen run.
REPO_DIR = None  # Automatically use a separate checkout for the resolved commit.
SETTINGS = {
    "backend": "transformers", "strategy": "full_history", "actor_prompt": "v3",
    "model_id": "Qwen/Qwen3-4B-Instruct-2507",
    "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "dtype": "float16", "recheck_cost": 2, "action_budget": 80,
    "num_locations": 1, "num_ingredients": 1,
    "context_limit": 8192, "action_max_new_tokens": 128, "generation_seed": 0,
}
FIRST_SEED = 0  # Matched to the audited oven-failure episode; only appliance instructions change.


## 1. Check out the selected source revision

An existing checkout is reused only if clean and already at the requested commit. No reset or overwrite is performed.

In [ ]:
if REPO_REF is None:
    REPO_REF = subprocess.check_output(
        ["git", "ls-remote", REPO_URL, "refs/heads/main"], text=True,
    ).split()[0]
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
    raise ValueError("REPO_REF must be a full implementation commit SHA.")
if REPO_DIR is None:
    REPO_DIR = Path("/content/64610-Memory-" + REPO_REF[:12])
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REF], check=True)
status = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
source_sha = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if status or source_sha != REPO_REF:
    raise RuntimeError("Existing checkout is dirty or at a different revision; preserve it and use a fresh REPO_DIR.")
print("Source commit:", source_sha)

## 2. Persist results before launching inference

Google Drive authorization happens through Colab. Every new launch directory preserves earlier attempts, including partial/error traces. Drive writes may add overhead to episode wall time; model latency is recorded separately.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PILOT_DIR = Path("/content/drive/MyDrive/64610-Memory/qwen-pilots") / uuid.uuid4().hex
PILOT_DIR.mkdir(parents=True, exist_ok=False)
print("Persistent pilot directory:", PILOT_DIR)

## 3. Install an isolated inference environment and run tests

The separate environment avoids conflicts with Colab's preinstalled Transformers, torchvision, or audio packages. This setup downloads the pinned CUDA-enabled PyTorch distribution; it may take several minutes. No model weights are loaded yet. Test output and the dependency freeze are saved in a fresh checks directory inside PILOT_DIR. The shared Python helper applies the Java/Py4J startup fix.

In [ ]:
import shutil

if shutil.which("java") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "openjdk-17-jre-headless"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.11"], check=True)
runtime = Path("/content/memory-qwen-venv")
if not runtime.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", str(runtime)], check=True)
PYTHON = str(runtime / "bin/python")
ENV = dict(os.environ, HF_HOME="/content/hf-cache")
subprocess.run([PYTHON, "-m", "pip", "install", "-r", "requirements-model.txt"], cwd=REPO_DIR, env=ENV, check=True)

def run_module(module, *arguments):
    result = subprocess.run([PYTHON, "-m", module, *map(str, arguments)],
                            cwd=REPO_DIR, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        print(result.stdout)
    result.check_returncode()
    return result.stdout

print(run_module("game_memory.pilot", "--output-dir", PILOT_DIR, "--check"))


## 4. Prepare the attempt without loading Qwen

Validate GPU support and save the game/model settings in an immutable manifest. Each attempt must use a fresh PILOT_DIR. The selected one-room/one-ingredient game is logged explicitly; it is not mixed with previous results.


In [ ]:
def run_diagnostic(seeds, *, prepare_only=False):
    arguments = []
    for key, value in SETTINGS.items():
        arguments.extend(["--" + key.replace("_", "-"), str(value)])
    arguments.extend(["--output-dir", str(PILOT_DIR), "--seeds", *map(str, seeds)])
    if prepare_only:
        return run_module("game_memory.pilot", *arguments, "--prepare-only")
    try:
        return session.run(arguments)
    finally:
        shutil.copyfile(SESSION_LOG, PILOT_DIR / "model-session.log")

print(run_diagnostic([FIRST_SEED], prepare_only=True))


## 5. Load Qwen once into the isolated Python process

This cell reuses a compatible live model, or closes the previous session before loading changed code/model settings. The notebook kernel keeps only a small connection to the isolated process, so its preinstalled packages cannot conflict with Qwen's environment. Only weights/tokenizer are reused: every episode creates a new game, memory, and usage ledger. Exact inputs/outputs are logged as before.

Editing game settings or selecting an existing prompt version does not require reloading weights. Every Run all launch creates a fresh attempt; weights are reused only when the recorded code commit and model settings match.


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from game_memory.colab_session import ColabSession

MODEL_SETTINGS = {
    "model_id": SETTINGS["model_id"], "revision": SETTINGS["model_revision"],
    "dtype": SETTINGS["dtype"], "context_limit": SETTINGS["context_limit"],
    "action_max_new_tokens": SETTINGS["action_max_new_tokens"],
    "generation_seed": SETTINGS["generation_seed"],
}
if "session" in globals() and session.process.poll() is None:
    if (session.ready["source"]["commit"] != source_sha
            or globals().get("LOADED_MODEL_SETTINGS") != MODEL_SETTINGS):
        session.close()
        print("Closed the previous model session because code or model settings changed.")
try:
    if "session" not in globals() or session.process.poll() is not None:
        SESSION_LOG = PILOT_DIR.parent / ("model-session-" + uuid.uuid4().hex + ".log")
        session = ColabSession(PYTHON, REPO_DIR, MODEL_SETTINGS, SESSION_LOG)
        LOADED_MODEL_SETTINGS = dict(MODEL_SETTINGS)
        print(session.ready)
    else:
        print("Reusing the loaded model; this episode still starts with fresh game state.")
finally:
    shutil.copyfile(SESSION_LOG, PILOT_DIR / "model-session.log")


## 6. Run one real episode using the loaded model

Run this cell once for this attempt. A game loss is saved as a completed execution with success=false; infrastructure failures remain errors. The live session avoids another weight load, while the episode starts with fresh state. Output is written to model-session.log and the episode trace.


In [ ]:
print(run_diagnostic([FIRST_SEED]))


## 7. Audit the checkpoint and inspect the trace

The report independently checks budgets, tokens, request/response pairing, labels, and completion. It keeps infrastructure errors separate from gameplay losses. A technically valid report does not demonstrate adequate gameplay capability. Open the saved events.jsonl to inspect exact prompts and decisions.

In [ ]:
output = run_module("game_memory.summarize_diagnostics", PILOT_DIR)
(PILOT_DIR / "checkpoint-report.json").write_text(output)
report = json.loads(output)
for result in report["episodes"]:
    print(json.dumps({key: result[key] for key in
        ("episode_id", "actor_prompt", "seed", "status", "success", "termination_reason", "steps",
         "recipe_rechecks", "budget_spent", "input_tokens", "output_tokens", "runtime_seconds")}, indent=2))
    print("Maximum score:", result["behavior"]["maximum_observed_score"])
print("Game settings:", json.loads((PILOT_DIR / "pilot-manifest.json").read_text())["episode"]["game"])
print("Logging checks passed:", report["logging_checks_passed"])
print("Infrastructure checks passed:", report["infrastructure_checks_passed"])


## Save the checkpoint for review

Download one bundle containing the manifest, dependency versions, checks, launch logs, result, and exact trace. Return that bundle for inspection before running memory comparisons. A completed execution is not evidence that the model can play successfully. Do not rerun the same seed in this directory; every attempt needs its own preserved folder.

In [ ]:
results = [json.loads(path.read_text()) for path in PILOT_DIR.glob("*/result.json")]
if len(results) != 1:
    raise RuntimeError("This checkpoint expects exactly one episode. Inspect the saved files.")
trace_path = PILOT_DIR / results[0]["episode_id"] / "events.jsonl"
print("Saved trace:", trace_path)


The archive contains research logs only; cached model weights are outside PILOT_DIR.

In [ ]:
from google.colab import files
archive = shutil.make_archive(str(PILOT_DIR.parent / (PILOT_DIR.name + "-checkpoint")), "zip", PILOT_DIR)
files.download(archive)


## Review before the next run

Return this attempt's ZIP for comparison with the already-audited oven failure. Inspect preparation, eating, ingredient handling, repairs, loops, and costs. A successful one-ingredient game establishes a small capability baseline; it may offer too little memory pressure for the research question.

**For one explicitly chosen next development run:** edit SETTINGS/FIRST_SEED in the configuration cell, rerun section 2 to create a fresh PILOT_DIR, rerun section 4, then sections 6–7 and the download cells. Skip setup/tests (section 3) and model loading (section 5) while code, dependencies, and model settings remain unchanged. No further run is selected or launched automatically. Increase difficulty one step at a time only after inspecting real gameplay; do not select seeds by which ones succeed.

Model weights and tokenizer remain in GPU memory, but history, game state, and accounting never carry across episodes. KV caches are per generation call. If you edit repository code, the worker refuses stale runs: use a new committed checkout. Run all performs the checks and replaces incompatible loaded sessions. Tests must run for code/dependency changes. A saved Colab copy keeps its own SETTINGS; reopen the current GitHub notebook to pick up code/configuration updates.

When finished, run `session.close()` in a new cell, then disconnect the GPU. Colab can still reclaim the runtime: Drive contains logs and Hugging Face caches contain weight files, but neither restores a live GPU process or an interrupted game. Every attempt is preserved rather than resumed midway.

This notebook's session path is tested locally with tiny random weights; the production Qwen run and Colab integration still require you. No eight-episode memory pilot runs here. Hybrid, the 120-episode study, dashboards, batch/resume, and further optimization remain deferred.
